# Phase 2.2: v2a-RSN Depth Selection

Apply depth-selection rules to v2a-RSN c-GC and c-GC-star outputs in two layers:

1. Descriptive depth selection from the observed instability trajectory `D_p`.
2. Bootstrap-calibrated depth selection when Phase 1 moving-block bootstrap outputs are available.

For each fish + method combination:
- Load `D_p` instability values from Phase 0 `summary.json` files.
- Apply absolute threshold and relative-drop rules as descriptive summaries.
- Load Phase 1 bootstrap pointwise bands when present.
- Apply the bootstrap-band rule and report global null-calibration statistics.
- Store descriptive and calibrated results in structured CSV and JSON outputs.
- Generate observed and calibrated visualization panels.

**Output:**
- `outputs/v2a-RSNs/n25-e9-r16/depth_selection/depth_selection_summary.csv`
- `outputs/v2a-RSNs/n25-e9-r16/depth_selection/depth_selection_calibrated_summary.csv`
- `outputs/v2a-RSNs/n25-e9-r16/depth_selection/depth_selection.json`
- `outputs/v2a-RSNs/n25-e9-r16/depth_selection/depth_selection_plot.png`
- `outputs/v2a-RSNs/n25-e9-r16/depth_selection/depth_selection_calibrated_plot.png`
- `outputs/v2a-RSNs/n25-e9-r16/depth_selection/manifest.json`

In [ ]:
from __future__ import annotations

import sys
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set up project paths
CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find '{CAUSALISED_GC_RELATIVE_PATH}' from {Path.cwd().resolve()}"
    )

# Add to path for imports
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.depth_selection import DepthSelector
from markovianity_diagnostic.experiments.v2a_rsn_utils import (
    V2A_ANALYSIS_PROFILE,
)

logger.info(f'Project root: {PROJECT_ROOT}')
print(f'Project root: {PROJECT_ROOT}')

V2A_OUTPUT_DIR = (
    PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / V2A_ANALYSIS_PROFILE
)
V2A_CALIBRATION_DIR = (
    PROJECT_ROOT / 'outputs' / 'calibration' / 'v2a' / V2A_ANALYSIS_PROFILE
)


In [ ]:
# Check for cached outputs
output_dir = V2A_OUTPUT_DIR / 'depth_selection'

expected_outputs = {
    'depth_selection_summary.csv': output_dir / 'depth_selection_summary.csv',
    'depth_selection_calibrated_summary.csv': output_dir / 'depth_selection_calibrated_summary.csv',
    'depth_selection.json': output_dir / 'depth_selection.json',
    'depth_selection_plot.png': output_dir / 'depth_selection_plot.png',
    'depth_selection_calibrated_plot.png': output_dir / 'depth_selection_calibrated_plot.png',
    'manifest.json': output_dir / 'manifest.json',
}

outputs_exist = all(fpath.exists() for fpath in expected_outputs.values())

if outputs_exist:
    logger.info("Outputs already exist - rerunning to reflect current upstream summaries")
    print("Outputs already exist - rerunning to reflect current upstream summaries")
    print(f"Output directory: {output_dir}")
    for name, path in expected_outputs.items():
        size_mb = path.stat().st_size / (1024 * 1024)
        logger.info(f"  {name} ({size_mb:.2f} MB)")
        print(f"  {name} ({size_mb:.2f} MB)")
else:
    logger.info("No complete cached output set - will run depth selection")
    print("No complete cached output set - will run depth selection")
    print(f"Output directory: {output_dir}")


In [ ]:
logger.info("Loading v2a-RSN summary data...")
start_load = time.time()

# Paths to analysis outputs
c_gc_summary_path = V2A_OUTPUT_DIR / 'c-GC' / 'summary.json'
c_gc_star_summary_path = V2A_OUTPUT_DIR / 'c-GC-star' / 'summary.json'

missing_summary_paths = [
    path for path in [c_gc_summary_path, c_gc_star_summary_path]
    if not path.exists()
]
if missing_summary_paths:
    raise FileNotFoundError(
        "Missing Phase 0 summary outputs: "
        + ", ".join(str(path) for path in missing_summary_paths)
    )

# Load summary data
with open(c_gc_summary_path) as f:
    c_gc_data = json.load(f)

with open(c_gc_star_summary_path) as f:
    c_gc_star_data = json.load(f)

load_elapsed = time.time() - start_load
logger.info(f"Loaded c-GC data: {len(c_gc_data)} recordings (elapsed: {load_elapsed:.2f}s)")
logger.info(f"Loaded c-GC-star data: {len(c_gc_star_data)} recordings")
print(f"Loaded c-GC data: {len(c_gc_data)} recordings")
print(f"Loaded c-GC-star data: {len(c_gc_star_data)} recordings")

# Show structure
print(f"\nFirst recording (c-GC):")
print(f"  Keys: {c_gc_data[0].keys()}")


In [ ]:
logger.info("Extracting fish IDs...")

# Create recording-level fish mapping.
# The same biological F-label can appear in more than one recording, so labels
# are assigned by recording order rather than by the embedded F# suffix.
c_gc_datasets = [entry['dataset'] for entry in c_gc_data]
c_gc_star_datasets = [entry['dataset'] for entry in c_gc_star_data]
if len(c_gc_datasets) != len(set(c_gc_datasets)):
    raise ValueError(f'Duplicate c-GC datasets are not supported: {c_gc_datasets}')
if set(c_gc_datasets) != set(c_gc_star_datasets):
    raise ValueError(
        'c-GC and c-GC-star summaries must contain the same recordings; '
        f'got c-GC={c_gc_datasets}, c-GC-star={c_gc_star_datasets}'
    )
fish_mapping = {
    dataset: f'fish-{index}'
    for index, dataset in enumerate(c_gc_datasets, start=1)
}
fish_ids = [fish_mapping[dataset] for dataset in c_gc_datasets]

logger.info(f"Fish mapping: {fish_mapping}")
print(f"Fish mapping: {fish_mapping}")

# Map each entry
for entry in c_gc_data:
    entry['fish'] = fish_mapping[entry['dataset']]
    
for entry in c_gc_star_data:
    entry['fish'] = fish_mapping[entry['dataset']]

logger.info(f"Mapped fish for c-GC: {[e['fish'] for e in c_gc_data]}")
logger.info(f"Mapped fish for c-GC-star: {[e['fish'] for e in c_gc_star_data]}")
print(f"\nMapped fish for c-GC: {[e['fish'] for e in c_gc_data]}")
print(f"Mapped fish for c-GC-star: {[e['fish'] for e in c_gc_star_data]}")


## Descriptive Depth Selection

This section uses only the observed `D_p` trajectory from Phase 0. The absolute and relative rules are descriptive and do not rely on the bootstrap null.

In [ ]:
logger.info("Applying descriptive depth selection rules...")
start_selection = time.time()

selector = DepthSelector()


def _as_depth_float_dict(values: dict) -> dict[int, float]:
    return {int(k): float(v) for k, v in values.items()}


def _descriptive_selection(entry: dict, method: str) -> dict:
    fish = entry['fish']
    dataset = entry['dataset']
    D_p = _as_depth_float_dict(entry.get('D_p', {}))
    result = selector.apply_all_rules(
        D_p=D_p,
        D_boot_pointwise={},
        epsilon=0.01,
        k_stable=2,
        fraction=0.1,
        confidence=0.95,
    )
    descriptive_warnings = [
        warning for warning in result.warnings
        if 'bootstrap' not in warning.lower()
    ]
    return {
        'fish': fish,
        'method': method,
        'dataset': dataset,
        'p_values': result.p_values,
        'D_p': D_p,
        'selected_absolute': result.selected['absolute'],
        'selected_relative': result.selected['relative'],
        'selected_bootstrap': None,
        'warnings': descriptive_warnings,
        'calibrated_warnings': [],
        'calibration_status': 'pending_bootstrap_lookup',
        'bootstrap_replicates': 0,
        'bootstrap_payload_path': None,
        'global_p_value': None,
        'global_T_obs': None,
        'global_critical_95': None,
        'reject_global_95': None,
        'first_exceedance_depth': None,
        'pointwise_exceedance_depths': [],
        'D_boot_pointwise': {},
    }


# Store results
results_list = []

# Process c-GC with progress tracking
logger.info("Processing c-GC recordings...")
for idx, entry in enumerate(c_gc_data, 1):
    logger.info(f"  [{idx}/{len(c_gc_data)}] Processing {entry['dataset']}")
    results_list.append(_descriptive_selection(entry, 'c-GC'))

# Process c-GC-star with progress tracking
logger.info("Processing c-GC-star recordings...")
for idx, entry in enumerate(c_gc_star_data, 1):
    logger.info(f"  [{idx}/{len(c_gc_star_data)}] Processing {entry['dataset']}")
    results_list.append(_descriptive_selection(entry, 'c-GC-star'))

selection_elapsed = time.time() - start_selection
logger.info(
    f"Applied descriptive depth selection to {len(results_list)} "
    f"fish-method combinations (elapsed: {selection_elapsed:.2f}s)"
)
print(f"Applied descriptive depth selection to {len(results_list)} fish-method combinations")


## Bootstrap-Calibrated Depth Selection

This section consumes Phase 1 v2a bootstrap outputs when they are available. Missing or incomplete bootstrap artifacts leave the descriptive results intact and mark the row as `missing_bootstrap` rather than silently treating the result as calibrated.

In [ ]:
logger.info("Loading bootstrap calibration bands...")
start_calibration_lookup = time.time()


def _optional_float(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    return float(value)


def _optional_int(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    return int(value)


def _optional_bool(value):
    if value is None:
        return None
    return bool(value)


def _load_bootstrap_payload(recording: str, method: str) -> tuple[dict | None, Path | None]:
    """Load split Phase 1 calibration payload for one recording-method pair."""
    run_payload_path = V2A_CALIBRATION_DIR / recording / method / 'bootstrap.json'
    if run_payload_path.exists():
        with open(run_payload_path) as f:
            return json.load(f), run_payload_path

    aggregate_payload_path = V2A_CALIBRATION_DIR / recording / 'bootstrap_results.json'
    if aggregate_payload_path.exists():
        with open(aggregate_payload_path) as f:
            aggregate_payload = json.load(f)
        payload = (
            aggregate_payload
            .get('methods', {})
            .get(method, {})
            .get(recording)
        )
        if payload is not None:
            return payload, aggregate_payload_path

    return None, None


def _normalize_pointwise_band(payload: dict | None) -> dict[int, dict[str, float]]:
    if not payload:
        return {}
    pointwise = payload.get('null', {}).get('pointwise', {})
    normalized = {}
    for depth, band in pointwise.items():
        if not isinstance(band, dict):
            continue
        normalized[int(depth)] = {
            key: float(value)
            for key, value in band.items()
            if value is not None
        }
    return normalized


def _pointwise_exceedances(D_p: dict[int, float], band: dict[int, dict[str, float]]) -> list[int]:
    exceedances = []
    for depth in sorted(D_p):
        if depth not in band:
            continue
        threshold = band[depth].get('critical_95', band[depth].get('upper'))
        if threshold is not None and D_p[depth] > threshold:
            exceedances.append(depth)
    return exceedances


calibration_status_counts = {}
for result in results_list:
    payload, payload_path = _load_bootstrap_payload(result['dataset'], result['method'])
    pointwise_band = _normalize_pointwise_band(payload)
    if not pointwise_band:
        result['calibration_status'] = 'missing_bootstrap'
        result['calibrated_warnings'] = [
            'Phase 1 bootstrap pointwise bands were not found for this recording-method pair.'
        ]
        calibration_status_counts[result['calibration_status']] = (
            calibration_status_counts.get(result['calibration_status'], 0) + 1
        )
        continue

    calibrated_selection = selector.apply_all_rules(
        D_p=result['D_p'],
        D_boot_pointwise=pointwise_band,
        epsilon=0.01,
        k_stable=2,
        fraction=0.1,
        confidence=0.95,
    )
    null = payload.get('null', {})
    observed = payload.get('observed', {})
    diagnosis = payload.get('diagnosis', {})
    metadata = payload.get('metadata', {})

    result['selected_bootstrap'] = calibrated_selection.selected['bootstrap_band']
    result['calibrated_warnings'] = [
        warning for warning in calibrated_selection.warnings
        if 'bootstrap' in warning.lower()
    ]
    result['calibration_status'] = 'bootstrap_calibrated'
    result['bootstrap_replicates'] = int(null.get('B') or metadata.get('B') or 0)
    result['bootstrap_payload_path'] = (
        str(payload_path.relative_to(PROJECT_ROOT)) if payload_path else None
    )
    result['global_p_value'] = _optional_float(null.get('p_value'))
    result['global_T_obs'] = _optional_float(observed.get('T_obs'))
    result['global_critical_95'] = _optional_float(null.get('critical_95'))
    result['reject_global_95'] = _optional_bool(diagnosis.get('reject_global_95'))
    result['first_exceedance_depth'] = _optional_int(
        diagnosis.get('first_exceedance_depth')
    )
    result['pointwise_exceedance_depths'] = _pointwise_exceedances(
        result['D_p'],
        pointwise_band,
    )
    result['D_boot_pointwise'] = pointwise_band
    calibration_status_counts[result['calibration_status']] = (
        calibration_status_counts.get(result['calibration_status'], 0) + 1
    )

calibration_elapsed = time.time() - start_calibration_lookup
logger.info(
    f"Bootstrap calibration lookup completed in {calibration_elapsed:.2f}s: "
    f"{calibration_status_counts}"
)
print("Bootstrap calibration status:")
for status, count in sorted(calibration_status_counts.items()):
    print(f"  {status}: {count}")


In [ ]:
logger.info("Creating summary dataframe...")

# Create summary dataframe
summary_data = []
for result in results_list:
    summary_data.append({
        'fish': result['fish'],
        'method': result['method'],
        'dataset': result['dataset'],
        'p_selected_absolute': result['selected_absolute'],
        'p_selected_relative': result['selected_relative'],
        'p_selected_bootstrap': result['selected_bootstrap'],
        'calibration_status': result['calibration_status'],
        'bootstrap_B': result['bootstrap_replicates'],
        'global_T_obs': result['global_T_obs'],
        'global_critical_95': result['global_critical_95'],
        'global_p_value': result['global_p_value'],
        'reject_global_95': result['reject_global_95'],
        'first_exceedance_depth': result['first_exceedance_depth'],
        'pointwise_exceedance_depths': ','.join(
            str(depth) for depth in result['pointwise_exceedance_depths']
        ),
        'max_depth': max(result['p_values']) if result['p_values'] else None,
        'max_D_p': max(result['D_p'].values()) if result['D_p'] else None,
        'n_warnings_descriptive': len(result['warnings']),
        'n_warnings_calibrated': len(result['calibrated_warnings']),
        'bootstrap_payload_path': result['bootstrap_payload_path'],
    })

summary_df = pd.DataFrame(summary_data)

logger.info(f"Summary shape: {summary_df.shape}")
logger.info("Depth Selection Summary:\n" + summary_df.to_string())
print("Depth Selection Summary:")
print(summary_df.to_string())

logger.info(f"\nSummary shape: {summary_df.shape}")
logger.info(f"Expected: {len(fish_ids)} fish x 2 methods = {len(fish_ids) * 2} rows")
print(f"\nSummary shape: {summary_df.shape}")
print(f"Expected: {len(fish_ids)} fish x 2 methods = {len(fish_ids) * 2} rows")


In [ ]:
logger.info("Exporting results...")
start_export = time.time()

# Create output directory
output_dir = V2A_OUTPUT_DIR / 'depth_selection'
output_dir.mkdir(parents=True, exist_ok=True)

# Save summary CSVs
csv_path = output_dir / 'depth_selection_summary.csv'
summary_df.to_csv(csv_path, index=False)
logger.info(f"Saved summary CSV to: {csv_path}")
print(f"Saved summary CSV to: {csv_path}")

calibrated_columns = [
    'fish',
    'method',
    'dataset',
    'p_selected_bootstrap',
    'calibration_status',
    'bootstrap_B',
    'global_T_obs',
    'global_critical_95',
    'global_p_value',
    'reject_global_95',
    'first_exceedance_depth',
    'pointwise_exceedance_depths',
    'bootstrap_payload_path',
]
calibrated_csv_path = output_dir / 'depth_selection_calibrated_summary.csv'
summary_df[calibrated_columns].to_csv(calibrated_csv_path, index=False)
logger.info(f"Saved calibrated summary CSV to: {calibrated_csv_path}")
print(f"Saved calibrated summary CSV to: {calibrated_csv_path}")


def _json_band(pointwise: dict[int, dict[str, float]]) -> dict[str, dict[str, float]]:
    return {
        str(int(depth)): {key: float(value) for key, value in band.items()}
        for depth, band in sorted(pointwise.items())
    }


# Save detailed JSON
json_output = {
    'metadata': {
        'analysis': 'v2a-RSN depth selection (Phase 2.2)',
        'analysis_profile': V2A_ANALYSIS_PROFILE,
        'selection_layers': ['descriptive', 'bootstrap_calibrated'],
        'rules': ['absolute', 'relative', 'bootstrap_band'],
        'absolute_epsilon': 0.01,
        'absolute_k_stable': 2,
        'relative_fraction': 0.1,
        'bootstrap_confidence': 0.95,
        'calibration_output_dir': str(V2A_CALIBRATION_DIR.relative_to(PROJECT_ROOT)),
        'n_fish': len(fish_ids),
        'n_methods': 2,
        'n_recordings': len(results_list),
        'calibration_status_counts': {
            str(key): int(value)
            for key, value in summary_df['calibration_status'].value_counts().items()
        },
    },
    'fish_mapping': fish_mapping,
    'results': [
        {
            'fish': r['fish'],
            'method': r['method'],
            'dataset': r['dataset'],
            'p_values': r['p_values'],
            'D_p': {str(k): float(v) for k, v in r['D_p'].items()},
            'selected': {
                'absolute': r['selected_absolute'],
                'relative': r['selected_relative'],
                'bootstrap_band': r['selected_bootstrap'],
            },
            'calibration': {
                'status': r['calibration_status'],
                'bootstrap_B': r['bootstrap_replicates'],
                'payload_path': r['bootstrap_payload_path'],
                'T_obs': r['global_T_obs'],
                'critical_95': r['global_critical_95'],
                'p_value': r['global_p_value'],
                'reject_global_95': r['reject_global_95'],
                'first_exceedance_depth': r['first_exceedance_depth'],
                'pointwise_exceedance_depths': r['pointwise_exceedance_depths'],
                'D_boot_pointwise': _json_band(r['D_boot_pointwise']),
            },
            'warnings': {
                'descriptive': r['warnings'],
                'calibrated': r['calibrated_warnings'],
            },
        }
        for r in results_list
    ],
}

json_path = output_dir / 'depth_selection.json'
with open(json_path, 'w') as f:
    json.dump(json_output, f, indent=2)
logger.info(f"Saved detailed JSON to: {json_path}")
print(f"Saved detailed JSON to: {json_path}")

manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'v2a_depth_selection',
    'analysis_profile': V2A_ANALYSIS_PROFILE,
    'status': 'complete',
    'input_paths': [
        str(c_gc_summary_path.relative_to(PROJECT_ROOT)),
        str(c_gc_star_summary_path.relative_to(PROJECT_ROOT)),
    ],
    'calibration_input_dir': str(V2A_CALIBRATION_DIR.relative_to(PROJECT_ROOT)),
    'output_paths': [
        str(csv_path.relative_to(PROJECT_ROOT)),
        str(calibrated_csv_path.relative_to(PROJECT_ROOT)),
        str(json_path.relative_to(PROJECT_ROOT)),
    ],
    'calibration_status_counts': json_output['metadata']['calibration_status_counts'],
}
manifest_path = output_dir / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
logger.info(f"Saved manifest to: {manifest_path}")
print(f"Saved manifest to: {manifest_path}")

export_elapsed = time.time() - start_export
logger.info(f"Export completed in {export_elapsed:.2f}s")


In [ ]:
logger.info("Generating descriptive depth selection plots...")
start_plot = time.time()

# Create descriptive depth selection plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

fish_list = sorted(set(r['fish'] for r in results_list))
colors = {'c-GC': 'tab:blue', 'c-GC-star': 'tab:orange'}
markers = {'c-GC': 'o', 'c-GC-star': 's'}

for idx, fish in enumerate(fish_list):
    logger.info(f"  Plotting {fish}...")
    ax = axes[idx]
    
    # Get results for this fish from both methods
    fish_results = [r for r in results_list if r['fish'] == fish]
    
    for result in fish_results:
        method = result['method']
        p_vals = sorted(result['p_values'])
        d_vals = [result['D_p'][p] for p in p_vals]
        
        # Plot D_p trajectory
        ax.plot(
            p_vals, d_vals,
            marker=markers[method],
            color=colors[method],
            linestyle='-',
            linewidth=2,
            markersize=8,
            label=method,
        )
        
        # Mark descriptive selected depths with vertical lines
        abs_sel = result['selected_absolute']
        rel_sel = result['selected_relative']
        
        if abs_sel is not None:
            ax.axvline(abs_sel, color=colors[method], linestyle='--', alpha=0.45, linewidth=1)
        if rel_sel is not None and rel_sel != abs_sel:
            ax.axvline(rel_sel, color=colors[method], linestyle=':', alpha=0.45, linewidth=1)
    
    ax.set_xlabel('Conditioning depth p', fontsize=11)
    ax.set_ylabel('Instability D_p', fontsize=11)
    ax.set_title(f'{fish}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(1, 8))
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

for ax in axes[len(fish_list):]:
    ax.axis('off')

plt.tight_layout()
png_path = output_dir / 'depth_selection_plot.png'
plt.savefig(png_path, dpi=200, bbox_inches='tight')
plot_elapsed = time.time() - start_plot
logger.info(f"Saved descriptive depth selection plot to: {png_path} (elapsed: {plot_elapsed:.2f}s)")
print(f"Saved descriptive depth selection plot to: {png_path}")
plt.show()


In [ ]:
logger.info("Generating calibrated depth selection plots...")
start_calibrated_plot = time.time()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, fish in enumerate(fish_list):
    ax = axes[idx]
    fish_results = [r for r in results_list if r['fish'] == fish]

    for result in fish_results:
        method = result['method']
        color = colors[method]
        p_vals = sorted(result['p_values'])
        d_vals = [result['D_p'][p] for p in p_vals]
        ax.plot(
            p_vals,
            d_vals,
            marker=markers[method],
            color=color,
            linestyle='-',
            linewidth=2,
            markersize=7,
            label=f'{method} observed',
        )

        pointwise = result['D_boot_pointwise']
        band_depths = [p for p in p_vals if p in pointwise]
        if band_depths:
            lower = [pointwise[p]['lower'] for p in band_depths]
            upper = [pointwise[p]['upper'] for p in band_depths]
            critical_95 = [pointwise[p].get('critical_95', pointwise[p]['upper']) for p in band_depths]
            ax.fill_between(
                band_depths,
                lower,
                upper,
                color=color,
                alpha=0.12,
                linewidth=0,
                label=f'{method} 95% null band',
            )
            ax.plot(
                band_depths,
                critical_95,
                color=color,
                linestyle=':',
                linewidth=1.2,
                alpha=0.8,
                label=f'{method} 95% critical',
            )

        boot_sel = result['selected_bootstrap']
        if boot_sel is not None:
            ax.axvline(boot_sel, color=color, linestyle='--', alpha=0.65, linewidth=1.4)

    ax.set_xlabel('Conditioning depth p', fontsize=11)
    ax.set_ylabel('Instability D_p', fontsize=11)
    ax.set_title(f'{fish}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(1, 8))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

for ax in axes[len(fish_list):]:
    ax.axis('off')

plt.tight_layout()
calibrated_png_path = output_dir / 'depth_selection_calibrated_plot.png'
plt.savefig(calibrated_png_path, dpi=200, bbox_inches='tight')
calibrated_plot_elapsed = time.time() - start_calibrated_plot
logger.info(
    f"Saved calibrated depth selection plot to: {calibrated_png_path} "
    f"(elapsed: {calibrated_plot_elapsed:.2f}s)"
)
print(f"Saved calibrated depth selection plot to: {calibrated_png_path}")
plt.show()

manifest['output_paths'].append(str(png_path.relative_to(PROJECT_ROOT)))
manifest['output_paths'].append(str(calibrated_png_path.relative_to(PROJECT_ROOT)))
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)


## Summary and Verification

In [ ]:
logger.info("\n" + "="*70)
logger.info("PHASE 2.2 DEPTH SELECTION SUMMARY")
logger.info("="*70)

print("\n" + "="*70)
print("PHASE 2.2 DEPTH SELECTION SUMMARY")
print("="*70)

logger.info(f"\nOutput files:")
logger.info(f"  CSV: {csv_path}")
logger.info(f"  Calibrated CSV: {calibrated_csv_path}")
logger.info(f"  JSON: {json_path}")
logger.info(f"  PNG: {png_path}")
logger.info(f"  Calibrated PNG: {calibrated_png_path}")
logger.info(f"  Manifest: {manifest_path}")
print(f"\nOutput files:")
print(f"  CSV: {csv_path}")
print(f"  Calibrated CSV: {calibrated_csv_path}")
print(f"  JSON: {json_path}")
print(f"  PNG: {png_path}")
print(f"  Calibrated PNG: {calibrated_png_path}")
print(f"  Manifest: {manifest_path}")

logger.info(f"\nData dimensions:")
logger.info(f"  N fish: {len(fish_ids)}")
logger.info(f"  N methods: 2 (c-GC, c-GC-star)")
logger.info(f"  Total rows (CSV): {len(summary_df)}")
print(f"\nData dimensions:")
print(f"  N fish: {len(fish_ids)}")
print(f"  N methods: 2 (c-GC, c-GC-star)")
print(f"  Total rows (CSV): {len(summary_df)}")

logger.info(f"\nDepth selection results:")
logger.info(f"  Absolute rule selected: {summary_df['p_selected_absolute'].notna().sum()} / {len(summary_df)}")
logger.info(f"  Relative rule selected: {summary_df['p_selected_relative'].notna().sum()} / {len(summary_df)}")
logger.info(f"  Bootstrap rule selected: {summary_df['p_selected_bootstrap'].notna().sum()} / {len(summary_df)}")
logger.info("Calibration status:\n" + summary_df['calibration_status'].value_counts().to_string())
print(f"\nDepth selection results:")
print(f"  Absolute rule selected: {summary_df['p_selected_absolute'].notna().sum()} / {len(summary_df)}")
print(f"  Relative rule selected: {summary_df['p_selected_relative'].notna().sum()} / {len(summary_df)}")
print(f"  Bootstrap rule selected: {summary_df['p_selected_bootstrap'].notna().sum()} / {len(summary_df)}")
print("\nCalibration status:")
print(summary_df['calibration_status'].value_counts().to_string())

logger.info(f"\nSelected depths:")
logger.info(
    summary_df[[
        'fish',
        'method',
        'p_selected_absolute',
        'p_selected_relative',
        'p_selected_bootstrap',
        'calibration_status',
        'global_p_value',
    ]].to_string(index=False)
)
print(f"\nSelected depths:")
print(
    summary_df[[
        'fish',
        'method',
        'p_selected_absolute',
        'p_selected_relative',
        'p_selected_bootstrap',
        'calibration_status',
        'global_p_value',
    ]].to_string(index=False)
)

logger.info(f"\nFile verification:")
print(f"\nFile verification:")
for fpath in [csv_path, calibrated_csv_path, json_path, png_path, calibrated_png_path, manifest_path]:
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        logger.info(f"  {fpath.name}: {size_mb:.2f} MB")
        print(f"  {fpath.name}: {size_mb:.2f} MB")
    else:
        logger.error(f"  {fpath.name}: MISSING")
        print(f"  {fpath.name}: MISSING")

logger.info(f"\n" + "="*70)
logger.info(f"Phase 2.2 complete. CSV has correct shape: {summary_df.shape[0] == len(fish_ids) * 2}")
print(f"\n" + "="*70)
print(f"Phase 2.2 complete. CSV has correct shape: {summary_df.shape[0] == len(fish_ids) * 2}")
